In [ ]:
import numpy as np
from scipy.ndimage import distance_transform_edt
import ipywidgets as widgets
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from pathlib import Path
import nibabel as nib
import blosc2

#### helper functions
def load_b2nd(path):
    """Load a .b2nd file as a numpy array using blosc2."""
    arr_obj = blosc2.open(path, mode="r")
    arr = np.asarray(arr_obj[:])
    return arr  # keep channels intact

def load_volume(path):
    """Load a 3D or 4D volume from NIfTI (.nii.gz), NPZ, NPY, or B2ND files."""
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"File not found: {path}")
    
    # Handle .nii.gz properly (suffix might be .gz, but we need to check the full extension)
    if path.name.endswith('.nii.gz'):
        return nib.load(str(path)).get_fdata()
    elif path.suffix == ".npy":
        return np.load(path)
    elif path.suffix == ".npz":
        data = np.load(path)
        key = list(data.keys())[0]
        return data[key]
    elif path.suffix == ".nii":
        return nib.load(str(path)).get_fdata()
    elif path.suffix == ".b2nd":
        return load_b2nd(path)
    else:
        raise ValueError(f"Unsupported format: {path.suffix} (full filename: {path.name})")

Distance map for the loss function:

In [ ]:


# ===========================
# Distance map computation
# ===========================

def compute_distance_maps(mask):
    """Compute distance transforms for a binary mask."""
    mask_bool = mask.astype(bool)
    dist_in = distance_transform_edt(mask_bool)
    dist_out = distance_transform_edt(~mask_bool)
    signed_dist = dist_in - dist_out
    return dist_in, dist_out, signed_dist

def build_distance_overlay(distance_slice, alpha=0.35, cmap='jet'):
    """RGBA overlay for distance map (signed)."""
    # Handle empty slices or constant values
    if distance_slice.max() == distance_slice.min():
        # Return transparent overlay if no variation
        overlay = np.zeros((*distance_slice.shape, 4))
        return overlay
    
    # Normalize for visualization only
    normed = (distance_slice - distance_slice.min()) / (distance_slice.max() - distance_slice.min() + 1e-6)
    
    # Get colormap
    if isinstance(cmap, str):
        cmap = plt.get_cmap(cmap)
    
    overlay = cmap(normed)
    overlay[..., 3] = alpha
    return overlay

# ===========================
# Interactive axial visualization
# ===========================

def visualize_distance_case(ct_volume, mask_volume, axis="axial"):
    """
    Interactive visualization of CT, mask, and distance maps.
    
    Parameters:
    -----------
    ct_volume : numpy array
        3D CT volume
    mask_volume : numpy array
        3D binary mask volume
    axis : str
        One of "axial", "coronal", or "sagittal"
    """
    # Ensure mask is binary
    if mask_volume.max() > 1:
        print("Warning: Mask contains values > 1. Converting to binary.")
        mask_volume = (mask_volume > 0).astype(np.uint8)
    
    # Compute distance maps
    dist_in, dist_out, signed_dist = compute_distance_maps(mask_volume)
    
    # Determine number of slices based on axis
    if axis == "axial":
        n_slices = ct_volume.shape[0]
    elif axis == "coronal":
        n_slices = ct_volume.shape[1]
    elif axis == "sagittal":
        n_slices = ct_volume.shape[2]
    else:
        raise ValueError(f"Invalid axis {axis}. Must be 'axial', 'coronal', or 'sagittal'")
    
    def plot_slice(idx):
        # Clear previous output
        plt.clf()
        
        # Extract the appropriate slice
        if axis == "axial":
            ct_slice = ct_volume[idx, :, :]
            mask_slice = mask_volume[idx, :, :]
            dist_in_slice = dist_in[idx, :, :]
            dist_out_slice = dist_out[idx, :, :]
            signed_slice = signed_dist[idx, :, :]
        elif axis == "coronal":
            ct_slice = ct_volume[:, idx, :]
            mask_slice = mask_volume[:, idx, :]
            dist_in_slice = dist_in[:, idx, :]
            dist_out_slice = dist_out[:, idx, :]
            signed_slice = signed_dist[:, idx, :]
        elif axis == "sagittal":
            ct_slice = ct_volume[:, :, idx]
            mask_slice = mask_volume[:, :, idx]
            dist_in_slice = dist_in[:, :, idx]
            dist_out_slice = dist_out[:, :, idx]
            signed_slice = signed_dist[:, :, idx]

        # Print statistics
        print(f"Slice {idx}:")
        print(f"  dist_in: max {dist_in_slice.max():.2f}, mean {dist_in_slice[dist_in_slice>0].mean() if np.any(dist_in_slice>0) else 0:.2f}")
        print(f"  dist_out: max {dist_out_slice.max():.2f}")
        print(f"  signed_dist: min {signed_slice.min():.2f}, max {signed_slice.max():.2f}")

        # Create mask overlay
        mask_overlay = np.zeros((*mask_slice.shape, 4))
        mask_overlay[mask_slice > 0] = [1, 0, 0, 0.35]  # Red with alpha
        
        # Create signed distance overlay
        signed_overlay = build_distance_overlay(signed_slice, alpha=0.35)

        # Create figure
        fig, axes = plt.subplots(1, 5, figsize=(20, 6))
        
        # CT
        im0 = axes[0].imshow(ct_slice, cmap="gray", origin="lower")
        axes[0].set_title("CT")
        axes[0].axis("off")
        
        # Mask overlay
        axes[1].imshow(ct_slice, cmap="gray", origin="lower")
        axes[1].imshow(mask_overlay, origin="lower")
        axes[1].set_title("Mask overlay")
        axes[1].axis("off")
        
        # dist_in
        im2 = axes[2].imshow(dist_in_slice, cmap="jet", origin="lower")
        axes[2].set_title("dist_in")
        plt.colorbar(im2, ax=axes[2], fraction=0.046, pad=0.04)
        axes[2].axis("off")
        
        # dist_out
        im3 = axes[3].imshow(dist_out_slice, cmap="jet", origin="lower")
        axes[3].set_title("dist_out")
        plt.colorbar(im3, ax=axes[3], fraction=0.046, pad=0.04)
        axes[3].axis("off")
        
        # signed distance overlay
        axes[4].imshow(ct_slice, cmap="gray", origin="lower")
        axes[4].imshow(signed_overlay, origin="lower")
        axes[4].set_title("Signed distance overlay")
        axes[4].axis("off")

        plt.tight_layout()
        plt.show()
    
    # Create interactive slider
    slice_slider = widgets.IntSlider(
        value=n_slices // 2, 
        min=0, 
        max=n_slices - 1, 
        step=1, 
        description=f"{axis.capitalize()} Slice:", 
        continuous_update=False,
        layout=widgets.Layout(width='80%')
    )
    
    # Display the interactive widget
    widgets.interact(plot_slice, idx=slice_slider)

# ===========================
# Run the visualization
# ===========================

# Update these paths to match your data
ct_path = "/data/colon_cancer/CC_Detection/raw_data/Dataset100_CC/imagesTr/4_0000.nii.gz"
mask_path = "/data/colon_cancer/CC_Detection/raw_data/Dataset100_CC/labelsTr/4.nii.gz"

print("Loading CT volume...")
ct = load_volume(ct_path)
print(f"CT volume shape: {ct.shape}, dtype: {ct.dtype}")

print("Loading mask volume...")
mask = load_volume(mask_path)
print(f"Mask volume shape: {mask.shape}, dtype: {mask.dtype}")

# Check if dimensions match
if ct.shape != mask.shape:
    print(f"WARNING: Shape mismatch! CT: {ct.shape}, Mask: {mask.shape}")
    # Try to align if possible (e.g., if mask has extra dimension)
    if len(mask.shape) > len(ct.shape) and mask.shape[-1] == 1:
        mask = mask.squeeze(-1)
        print(f"Squeezed mask to: {mask.shape}")
    elif len(ct.shape) > len(mask.shape) and ct.shape[-1] == 1:
        ct = ct.squeeze(-1)
        print(f"Squeezed CT to: {ct.shape}")

print("\nStarting interactive visualization...")
visualize_distance_case(ct, mask, axis="sagittal")

Test strategies for the normalization of the distance map:

In [ ]:

# ---------------------------------------------------
# Helpers
# ---------------------------------------------------

def normalize_global(arr):
    """Normalize array to [0, 1] globally (for visualization)."""
    arr_min, arr_max = arr.min(), arr.max()
    return (arr - arr_min) / (arr_max - arr_min + 1e-6)


def compute_dist_out(mask_volume):
    """
    mask_volume: [D, H, W], binary (1 = lesion, 0 = background)
    returns: dist_out [D, H, W]
    """
    mask_bool = mask_volume.astype(bool)
    dist_out = distance_transform_edt(~mask_bool)
    return dist_out


def create_bad_prediction(mask_volume, extra_blob_offset=(10, 10, 10), extra_radius=3):
    """
    Create a synthetic bad prediction:
      - Start from the true mask
      - Add a second small blob far away (simulating an extra component)
      - Return as soft probs in [0,1]
    """
    pred = mask_volume.astype(np.float32).copy()

    # find a voxel inside the lesion as center
    lesion_coords = np.argwhere(mask_volume > 0)
    if len(lesion_coords) == 0:
        raise ValueError("Mask has no foreground voxels.")
    center = lesion_coords[len(lesion_coords) // 2]

    # shift center by some offset
    zc, yc, xc = center + np.array(extra_blob_offset)
    zc = np.clip(zc, 0, mask_volume.shape[0] - 1)
    yc = np.clip(yc, 0, mask_volume.shape[1] - 1)
    xc = np.clip(xc, 0, mask_volume.shape[2] - 1)

    zz, yy, xx = np.ogrid[:mask_volume.shape[0], :mask_volume.shape[1], :mask_volume.shape[2]]
    extra_blob = (zz - zc) ** 2 + (yy - yc) ** 2 + (xx - xc) ** 2 <= extra_radius ** 2

    # Add extra blob with high probability
    pred[extra_blob] = 1.0

    # Optionally smooth / soften probabilities a bit
    return pred


# ---------------------------------------------------
# Visualization
# ---------------------------------------------------

def visualize_distance_weight_penalty(
    ct_volume,
    mask_volume,
    axis="axial",
    use_radius_band=False,
    R=5.0,
):
    """
    Visualize how distance-based weights w will multiply predicted probabilities.

    - dist_out: distance to nearest lesion voxel
    - weights w: normalized dist_out (or binary mask dist_out > R)
    - p_pred: fake bad prediction with extra blob
    - penalty: p_pred * w
    """
    dist_out = compute_dist_out(mask_volume)         
    #dist_out_norm = normalize_global(dist_out)       
    #dist_out_norm = np.sqrt(dist_out)
    dist_out_norm = np.log1p(dist_out)
    # Define weights:
    if use_radius_band:
        # Only penalize voxels beyond radius R
        w = (dist_out > R).astype(np.float32)
    else:
        # Linear distance weighting: farther → larger weight
        w = dist_out_norm.astype(np.float32)

    # Fake prediction (GT + extra blob)
    p_pred = create_bad_prediction(mask_volume)       

    # Distance-weighted penalty map
    penalty_map = p_pred * w                        

    # Choose axis
    if axis == "axial":
        n_slices = ct_volume.shape[0]
    elif axis == "coronal":
        n_slices = ct_volume.shape[1]
    elif axis == "sagittal":
        n_slices = ct_volume.shape[2]
    else:
        raise ValueError("axis must be axial/coronal/sagittal")

    def get_slice(vol, idx):
        if axis == "axial":
            return vol[idx]
        elif axis == "coronal":
            return vol[:, idx, :]
        else:
            return vol[:, :, idx]

    def plot_slice(idx):
        ct_slice   = get_slice(ct_volume, idx)
        mask_slice = get_slice(mask_volume, idx)
        dist_slice = get_slice(dist_out_norm, idx)
        w_slice    = get_slice(w, idx)
        p_slice    = get_slice(p_pred, idx)
        pen_slice  = get_slice(penalty_map, idx)

        plt.figure(figsize=(24, 5))

        # 1) CT + GT mask
        plt.subplot(1, 5, 1)
        plt.imshow(ct_slice, cmap="gray", origin="lower")
        mask_rgb = np.zeros((*mask_slice.shape, 4))
        mask_rgb[mask_slice > 0] = [1, 0, 0, 0.4]  # red
        plt.imshow(mask_rgb, origin="lower")
        plt.title("GT Mask Overlay")
        plt.axis("off")

        # 2) dist_out (normalized)
        plt.subplot(1, 5, 2)
        im2 = plt.imshow(dist_slice, cmap="jet", origin="lower")
        plt.title("dist_out (norm)")
        plt.colorbar(im2, fraction=0.046, pad=0.04)
        plt.axis("off")

        # 3) weight map w
        plt.subplot(1, 5, 3)
        im3 = plt.imshow(w_slice, cmap="magma", origin="lower", vmin=0, vmax=w_slice.max() + 1e-6)
        plt.title("Weight w (penalty weight)")
        plt.colorbar(im3, fraction=0.046, pad=0.04)
        plt.axis("off")

        # 4) fake prediction (p_pred)
        plt.subplot(1, 5, 4)
        im4 = plt.imshow(p_slice, cmap="viridis", origin="lower", vmin=0, vmax=1)
        plt.title("Fake Prediction p_pred")
        plt.colorbar(im4, fraction=0.046, pad=0.04)
        plt.axis("off")

        # 5) distance-weighted penalty p_pred * w
        plt.subplot(1, 5, 5)
        im5 = plt.imshow(pen_slice, cmap="inferno", origin="lower")
        plt.title("Penalty Map = p_pred * w")
        plt.colorbar(im5, fraction=0.046, pad=0.04)
        plt.axis("off")

        plt.tight_layout()
        plt.show()

    slice_slider = widgets.IntSlider(
        value=n_slices // 2, min=0, max=n_slices - 1,
        step=1, description="Slice", continuous_update=False
    )
    widgets.interact(plot_slice, idx=slice_slider)


# ---------------------------------------------------
# Example usage (adapt paths to your data)
# ---------------------------------------------------
ct_path = "/data/colon_cancer/CC_Detection/raw_data/Dataset100_CC/imagesTr/600_0000.nii.gz"
mask_path = "/data/colon_cancer/CC_Detection/raw_data/Dataset100_CC/labelsTr/600.nii.gz"
#
ct = load_volume(ct_path)      
mask = load_volume(mask_path)  
#
# # Visualize with linear distance weighting
visualize_distance_weight_penalty(ct, mask, axis="sagittal", use_radius_band=False)
#
# # Or visualize with hard radius band (only penalize outside R)
visualize_distance_weight_penalty(ct, mask, axis="sagittal", use_radius_band=True, R=5.0)